In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.data import load_data, split_by_year
from src.features import add_ratios
from src.config import FEATURES, GBM_PARAMS, RISK_TREND
from src.gbm import cv_lgb, fit_lgb
from src.scorecard import WOEBinner
from src.benchmark import zpp_components, fit_caps, altman_zpp, ZScoreToPD
from src.stability import psi, psi_table

In [2]:
train, val, test = (add_ratios(d) for d in split_by_year(load_data()))
splits = {"train": train, "val": val, "test": test}

In [3]:
# Z'' benchmark (notebook 03)
caps = fit_caps(zpp_components(train))
z_map = ZScoreToPD().fit(altman_zpp(train, caps), train["default"])

# WOE scorecard (notebook 04)
binner = WOEBinner(RISK_TREND).fit(train, train["default"])
scorecard = LogisticRegression(max_iter=1000, C=np.inf).fit(binner.transform(train), train["default"])

# LightGBM (notebook 05)
n_trees = int(cv_lgb(train, FEATURES)["trees"].median())
model = fit_lgb(train, None, FEATURES, {**GBM_PARAMS, "n_estimators": n_trees})
n_trees

279

In [4]:
split_pd = {
    name: {
        "z''": z_map.predict_proba(altman_zpp(d, caps)),
        "scorecard": scorecard.predict_proba(binner.transform(d))[:, 1],
        "lightgbm": model.predict_proba(d[FEATURES])[:, 1],
    }
    for name, d in splits.items()
}

rows = []
for split, d in splits.items():
    for name, p in split_pd[split].items():
        observed = d["default"].mean()
        rows.append({"model": name, "split": split, "mean_pd": p.mean(), "observed": observed, "ratio": p.mean() / observed})
pd.DataFrame(rows).set_index(["model", "split"]).round(4)

,,mean_pd,observed,ratio
model,split,,,
z'',train,0.0072,0.0072,1.0000
scorecard,train,0.0072,0.0072,0.9997
lightgbm,train,0.0072,0.0072,1.0000
z'',val,0.0076,0.0083,0.9167
scorecard,val,0.0065,0.0083,0.7781
lightgbm,val,0.0057,0.0083,0.6891
z'',test,0.0076,0.0097,0.7812
scorecard,test,0.0069,0.0097,0.7119
lightgbm,test,0.0070,0.0097,0.7175


In [5]:
for name in ["scorecard", "lightgbm"]:
    for split, d in splits.items():
        d[f"pd_{name}"] = split_pd[split][name]

cols = ["pd_scorecard", "pd_lightgbm"] + FEATURES
psi_table(train, {"val": val, "test": test}, cols).round(3)

,n_bins,missing_ref,psi_val,flag_val,missing_val,psi_test,flag_test,missing_test
pd_scorecard,10,0.0,0.006,stable,0.0,0.015,stable,0.0
pd_lightgbm,10,0.0,0.018,stable,0.0,0.010,stable,0.0
mve_tl,10,0.0,0.026,stable,0.0,0.021,stable,0.0
tl_ta,10,0.0,0.013,stable,0.0,0.036,stable,0.0
ni_ta,10,0.0,0.010,stable,0.0,0.008,stable,0.0
quick,10,0.0,0.007,stable,0.0,0.006,stable,0.0
re_ta,10,0.0,0.021,stable,0.0,0.017,stable,0.0
log_ta,10,0.0,0.086,stable,0.0,0.144,monitor,0.0
